# Assignment 07: IMDB Text Classification in Multiple Ways

**Student:** Amankeldy Abylaj


## Introduction
This notebook compares three sentiment-classification pipelines on IMDB movie reviews:
1. **Word-level integer encoding + learned embedding + GRU**
2. **Gensim Word2Vec embeddings + classical classifier**
3. **Character-level neural model**


In [ ]:
# Core imports
import os
import re
import math
import random
import string
from collections import Counter

import numpy as np
import pandas as pd

from datasets import load_dataset
from gensim.models import Word2Vec

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import spacy
from spacy.lang.en.stop_words import STOP_WORDS

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1053, in launch_instance
    app.start()
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", lin

Using device: cpu


## Part A. Load IMDB + Compare Preprocessing
We load IMDB, create train/validation/test splits, inspect examples, and compare two preprocessing variants:
- **Simple:** lowercase + punctuation removal + whitespace split
- **Advanced:** spaCy tokenizer + lowercase + alpha filtering + stopword removal

In [ ]:
# Load IMDB dataset (Hugging Face datasets)
imdb = load_dataset('imdb')

train_full_texts = list(imdb['train']['text'])
train_full_labels = list(imdb['train']['label'])
test_full_texts = list(imdb['test']['text'])
test_full_labels = list(imdb['test']['label'])

# Build a validation split from the original training set
idx = np.arange(len(train_full_texts))
np.random.shuffle(idx)

val_ratio = 0.2
val_size = int(len(idx) * val_ratio)
val_idx = idx[:val_size]
train_idx = idx[val_size:]

train_texts = [train_full_texts[i] for i in train_idx]
train_labels = [train_full_labels[i] for i in train_idx]
val_texts = [train_full_texts[i] for i in val_idx]
val_labels = [train_full_labels[i] for i in val_idx]

# Use a balanced test subset for faster end-to-end experiments
test_subset_per_class = 5000
test_pos_texts = [t for t, y in zip(test_full_texts, test_full_labels) if y == 1][:test_subset_per_class]
test_neg_texts = [t for t, y in zip(test_full_texts, test_full_labels) if y == 0][:test_subset_per_class]
test_texts = test_neg_texts + test_pos_texts
test_labels = [0] * len(test_neg_texts) + [1] * len(test_pos_texts)

print('Split sizes:')
print(f'Train: {len(train_texts)}')
print(f'Validation: {len(val_texts)}')
print(f'Test (subset): {len(test_texts)}')

def avg_len(texts):
    # Average number of whitespace-delimited tokens before preprocessing
    return np.mean([len(t.split()) for t in texts])

def class_balance(labels):
    c = Counter(labels)
    total = len(labels)
    return {k: (v, v / total) for k, v in c.items()}

print('\nAverage review length (raw whitespace tokens):')
print(f'Train: {avg_len(train_texts):.2f}')
print(f'Validation: {avg_len(val_texts):.2f}')
print(f'Test: {avg_len(test_texts):.2f}')

print('\nClass balance (label -> count, ratio):')
print('Train:', class_balance(train_labels))
print('Validation:', class_balance(val_labels))
print('Test:', class_balance(test_labels))

# Show 2 positive and 2 negative examples from train
pos_examples = [t for t, y in zip(train_texts, train_labels) if y == 1][:2]
neg_examples = [t for t, y in zip(train_texts, train_labels) if y == 0][:2]

print('\n2 Positive examples (truncated):')
for i, ex in enumerate(pos_examples, 1):
    print(f'POS {i}:', ex[:350].replace('\n', ' '), '...')

print('\n2 Negative examples (truncated):')
for i, ex in enumerate(neg_examples, 1):
    print(f'NEG {i}:', ex[:350].replace('\n', ' '), '...')

Split sizes:
Train: 20000
Validation: 5000
Test (subset): 10000

Average review length (raw whitespace tokens):
Train: 233.91
Validation: 233.28
Test: 229.20

Class balance (label -> count, ratio):
Train: {1: (9999, 0.49995), 0: (10001, 0.50005)}
Validation: {1: (2501, 0.5002), 0: (2499, 0.4998)}
Test: {0: (5000, 0.5), 1: (5000, 0.5)}

2 Positive examples (truncated):
POS 1: just saw this exquisite 1982 movie Return of the Soldier, based on Rebecca West's novel. Its about a shell-shocked fortyish Captain who doesn't even tell his wife he has returned to British soil, but remains in a hospital in London. He's lost his memory and is a boy again, with a lingering yen for the lower class sweetheart he pursued 25 years earl ...
POS 2: The Unborn tells the tale of a married couple named Virginia (Brooke Adames) & Bradley Marshall (Jeff Hayenga) who have tried for the last five years to conceive, Virginia has had two miscarriages since then & is desperate to have a child. They visit Dr. Richa

In [ ]:
# Preprocessing utilities
punct_table = str.maketrans('', '', string.punctuation)
nlp = spacy.blank('en')

def preprocess_simple(text):
    # Simple variant: lowercase, remove punctuation, whitespace tokenize
    text = text.lower().translate(punct_table)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

def preprocess_advanced(text):
    # Advanced variant: spaCy tokenizer + alpha tokens + stopword removal
    doc = nlp(text)
    tokens = []
    for tok in doc:
        t = tok.text.lower().strip()
        if not t:
            continue
        if not t.isalpha():
            continue
        if t in STOP_WORDS:
            continue
        tokens.append(t)
    return tokens

# Tokenize all splits with both variants
train_tokens_simple = [preprocess_simple(t) for t in train_texts]
val_tokens_simple = [preprocess_simple(t) for t in val_texts]
test_tokens_simple = [preprocess_simple(t) for t in test_texts]

train_tokens_adv = [preprocess_advanced(t) for t in train_texts]
val_tokens_adv = [preprocess_advanced(t) for t in val_texts]
test_tokens_adv = [preprocess_advanced(t) for t in test_texts]

def vocab_stats(tokenized_texts):
    flat = [tok for review in tokenized_texts for tok in review]
    c = Counter(flat)
    vocab_size = len(c)
    top10 = c.most_common(10)
    rare10 = sorted([w for w, n in c.items() if n == 1])[:10]
    return vocab_size, top10, rare10

simple_vocab_size, simple_top10, simple_rare10 = vocab_stats(train_tokens_simple)
adv_vocab_size, adv_top10, adv_rare10 = vocab_stats(train_tokens_adv)

print('Simple preprocessing:')
print('Vocab size before cutoff:', simple_vocab_size)
print('Top-10 frequent tokens:', simple_top10)
print('10 rare tokens:', simple_rare10)

print('\nAdvanced preprocessing:')
print('Vocab size before cutoff:', adv_vocab_size)
print('Top-10 frequent tokens:', adv_top10)
print('10 rare tokens:', adv_rare10)

print('\n5 tokenized examples (simple):')
for i in range(5):
    print(f'#{i+1}:', train_tokens_simple[i][:30])

print('\n5 tokenized examples (advanced):')
for i in range(5):
    print(f'#{i+1}:', train_tokens_adv[i][:30])

Simple preprocessing:
Vocab size before cutoff: 107393
Top-10 frequent tokens: [('the', 267798), ('and', 130168), ('a', 129250), ('of', 116302), ('to', 107875), ('is', 85367), ('in', 74494), ('it', 61585), ('i', 60676), ('this', 60160)]
10 rare tokens: ['\x10own', '000', '0000000000001', '000001', '00000110', '0001', '00015', '0010', '00383042', '0080']

Advanced preprocessing:
Vocab size before cutoff: 65945
Top-10 frequent tokens: [('movie', 34479), ('film', 31244), ('like', 16036), ('good', 11867), ('time', 9855), ('story', 9283), ('people', 7295), ('bad', 7279), ('great', 7142), ('way', 6229)]
10 rare tokens: ['aaaaah', 'aaaaatch', 'aaaahhhhhhh', 'aaaarrgh', 'aaah', 'aaaugh', 'aaawwwwnnn', 'aachen', 'aaghh', 'aah']

5 tokenized examples (simple):
#1: ['just', 'saw', 'this', 'exquisite', '1982', 'movie', 'return', 'of', 'the', 'soldier', 'based', 'on', 'rebecca', 'wests', 'novel', 'its', 'about', 'a', 'shellshocked', 'fortyish', 'captain', 'who', 'doesnt', 'even', 'tell', 'his', 'wi

### Interpretation (Part A)
In sentiment classification, preprocessing changes both information quality and vocabulary noise.
- Simple preprocessing keeps more words (including noisy tokens), often preserving sentiment markers like negations.
- Advanced preprocessing removes many function words/noise, which can improve signal-to-noise ratio but may also drop useful sentiment cues.
We will compare downstream models to see which trade-off works better.

## Part B. Word-Level Integer Encoding Model
We build a vocabulary from **train only**, add special tokens `<PAD>` and `<UNK>`, encode sequences, and train a GRU classifier.

In [ ]:
# Use simple preprocessing variant for the word-level model
word_train_tokens = train_tokens_simple
word_val_tokens = val_tokens_simple
word_test_tokens = test_tokens_simple

PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
PAD_IDX = 0
UNK_IDX = 1

min_freq = 3
max_len_word = 220

token_counter = Counter(tok for review in word_train_tokens for tok in review)
vocab_words = [w for w, c in token_counter.items() if c >= min_freq]
stoi = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
for w in sorted(vocab_words):
    stoi[w] = len(stoi)
itos = {i: w for w, i in stoi.items()}

def encode_and_pad(tokens, stoi_map, max_len):
    ids = [stoi_map.get(t, UNK_IDX) for t in tokens]
    length = min(len(ids), max_len)
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids = ids + [PAD_IDX] * (max_len - len(ids))
    return ids, length

X_train_word, L_train_word = zip(*[encode_and_pad(t, stoi, max_len_word) for t in word_train_tokens])
X_val_word, L_val_word = zip(*[encode_and_pad(t, stoi, max_len_word) for t in word_val_tokens])
X_test_word, L_test_word = zip(*[encode_and_pad(t, stoi, max_len_word) for t in word_test_tokens])

X_train_word = np.array(X_train_word, dtype=np.int64)
X_val_word = np.array(X_val_word, dtype=np.int64)
X_test_word = np.array(X_test_word, dtype=np.int64)

y_train_arr = np.array(train_labels, dtype=np.int64)
y_val_arr = np.array(val_labels, dtype=np.int64)
y_test_arr = np.array(test_labels, dtype=np.int64)

print('Word model setup:')
print('min_freq:', min_freq)
print('max sequence length:', max_len_word)
print('vocabulary size:', len(stoi))
print('padding strategy: post-padding to fixed max_len, truncation from the tail')

Word model setup:
min_freq: 3
max sequence length: 220
vocabulary size: 39311
padding strategy: post-padding to fixed max_len, truncation from the tail


In [ ]:
# Dataset for encoded word sequences
class WordDataset(Dataset):
    # Dataset for encoded word sequences
    def __init__(self, X, lengths, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.lengths[idx], self.y[idx]

class WordClassifier(nn.Module):
    # Embedding -> GRU -> classifier
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        _, h_n = self.rnn(packed)
        last_hidden = h_n[-1]
        return self.fc(last_hidden)

def run_epoch_word(model, loader, optimizer=None):
    # One epoch for train/eval
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for x, lengths, y in loader:
        x = x.to(device)
        lengths = lengths.to(device)
        y = y.to(device)

        if train_mode:
            optimizer.zero_grad()

        logits = model(x, lengths)
        loss = criterion(logits, y)

        if train_mode:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().tolist())
        all_targets.extend(y.detach().cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    return avg_loss, acc, all_preds, all_targets

batch_size = 128
train_loader_word = DataLoader(WordDataset(X_train_word, L_train_word, y_train_arr), batch_size=batch_size, shuffle=True)
val_loader_word = DataLoader(WordDataset(X_val_word, L_val_word, y_val_arr), batch_size=batch_size, shuffle=False)
test_loader_word = DataLoader(WordDataset(X_test_word, L_test_word, y_test_arr), batch_size=batch_size, shuffle=False)

word_model = WordClassifier(
    vocab_size=len(stoi),
    embed_dim=128,
    hidden_size=128,
    num_classes=2,
    pad_idx=PAD_IDX
).to(device)

optimizer = torch.optim.Adam(word_model.parameters(), lr=1e-3)
epochs_word = 4

for epoch in range(1, epochs_word + 1):
    train_loss, train_acc, _, _ = run_epoch_word(word_model, train_loader_word, optimizer=optimizer)
    val_loss, val_acc, _, _ = run_epoch_word(word_model, val_loader_word, optimizer=None)
    print(f'Epoch {epoch}/{epochs_word} | train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}')

_, train_acc_word, _, _ = run_epoch_word(word_model, train_loader_word, optimizer=None)
_, val_acc_word, _, _ = run_epoch_word(word_model, val_loader_word, optimizer=None)
_, test_acc_word, test_preds_word, test_targets_word = run_epoch_word(word_model, test_loader_word, optimizer=None)

print('\nWord-level model accuracy:')
print(f'Train accuracy: {train_acc_word:.4f}')
print(f'Validation accuracy: {val_acc_word:.4f}')
print(f'Test accuracy: {test_acc_word:.4f}')

Epoch 1/4 | train_loss=0.6716, train_acc=0.5834, val_loss=0.6699, val_acc=0.5932
Epoch 2/4 | train_loss=0.6589, train_acc=0.5979, val_loss=0.6557, val_acc=0.6032
Epoch 3/4 | train_loss=0.5140, train_acc=0.7467, val_loss=0.4669, val_acc=0.7784
Epoch 4/4 | train_loss=0.3311, train_acc=0.8580, val_loss=0.3960, val_acc=0.8262

Word-level model accuracy:
Train accuracy: 0.8998
Validation accuracy: 0.8262
Test accuracy: 0.8225


In [ ]:
# Show 3 correct and 3 incorrect predictions from test set
correct_idx = [i for i, (p, y) in enumerate(zip(test_preds_word, test_targets_word)) if p == y][:3]
wrong_idx = [i for i, (p, y) in enumerate(zip(test_preds_word, test_targets_word)) if p != y][:3]

print('3 correct predictions:')
for i in correct_idx:
    print(f'Index {i} | pred={test_preds_word[i]} | true={test_targets_word[i]}')
    print(test_texts[i][:280].replace('\n', ' '), '...')
    print('-' * 80)

print('\n3 incorrect predictions:')
for i in wrong_idx:
    print(f'Index {i} | pred={test_preds_word[i]} | true={test_targets_word[i]}')
    print(test_texts[i][:280].replace('\n', ' '), '...')
    print('-' * 80)

3 correct predictions:
Index 0 | pred=0 | true=0
I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets,  ...
--------------------------------------------------------------------------------
Index 1 | pred=0 | true=0
Worth the entertainment value of a rental, especially if you like action movies. This one features the usual car chases, fights with the great Van Damme kick style, shooting battles with the 40 shell load shotgun, and even terrorist style bombs. All of this is entertaining and co ...
--------------------------------------------------------------------------------
Index 2 | pred=0 | true=0
its a totally average film with a few semi-alright action sequences that make the plot seem a little better and remind the viewer of the classic van dam films. parts o

### Interpretation (Part B)
The GRU word-level model usually gives strong baseline performance because it learns task-specific embeddings and sequence order.
Typical errors come from sarcasm, long context dependencies, or mixed sentiment in one review.

## Part C. Gensim Embedding Model
We train Word2Vec on train tokens, create one vector per review via mean pooling, and compare at least two OOV strategies.

In [ ]:
# Train Word2Vec on training tokens (simple preprocessing)
w2v_dim = 100
w2v = Word2Vec(
    sentences=word_train_tokens,
    vector_size=w2v_dim,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10,
    seed=SEED
)

# Fixed random vector for optional OOV usage
rng = np.random.default_rng(SEED)
random_oov_vec = rng.normal(0, 0.1, size=(w2v_dim,))

# Precompute a global average vector for fallback
global_avg_vec = np.mean(w2v.wv.vectors, axis=0)

def review_to_vec(tokens, strategy='skip'):
    # Convert one token list to one dense vector with OOV handling
    vecs = []
    found = 0
    missing = 0

    for t in tokens:
        if t in w2v.wv:
            vecs.append(w2v.wv[t])
            found += 1
        else:
            missing += 1
            if strategy == 'skip':
                continue
            elif strategy == 'avg':
                vecs.append(global_avg_vec)
            elif strategy == 'random':
                vecs.append(random_oov_vec)

    if len(vecs) == 0:
        return np.zeros(w2v_dim, dtype=np.float32), found, missing

    # Mean pooling review representation
    return np.mean(np.array(vecs), axis=0), found, missing

def build_matrix(tokenized_texts, strategy):
    X = []
    total_found, total_missing = 0, 0
    for toks in tokenized_texts:
        vec, found, missing = review_to_vec(toks, strategy=strategy)
        X.append(vec)
        total_found += found
        total_missing += missing
    return np.array(X), total_found, total_missing

strategies = ['skip', 'avg']
gensim_results = []

for strategy in strategies:
    X_train_g, found_train, miss_train = build_matrix(word_train_tokens, strategy)
    X_val_g, found_val, miss_val = build_matrix(word_val_tokens, strategy)
    X_test_g, found_test, miss_test = build_matrix(word_test_tokens, strategy)

    clf = LogisticRegression(max_iter=1200, random_state=SEED)
    clf.fit(X_train_g, y_train_arr)

    pred_train = clf.predict(X_train_g)
    pred_val = clf.predict(X_val_g)
    pred_test = clf.predict(X_test_g)

    acc_train = accuracy_score(y_train_arr, pred_train)
    acc_val = accuracy_score(y_val_arr, pred_val)
    acc_test = accuracy_score(y_test_arr, pred_test)

    gensim_results.append({
        'strategy': strategy,
        'train_acc': acc_train,
        'val_acc': acc_val,
        'test_acc': acc_test,
        'found_tokens_total': found_train + found_val + found_test,
        'missing_tokens_total': miss_train + miss_val + miss_test
    })

gensim_df = pd.DataFrame(gensim_results).sort_values('val_acc', ascending=False).reset_index(drop=True)
print(gensim_df)

  strategy  train_acc  val_acc  test_acc  found_tokens_total  \
0      avg    0.86535   0.8642    0.8547             7977119   
1     skip    0.86480   0.8638    0.8553             7977119   

   missing_tokens_total  
0                128255  
1                128255  


### Interpretation (Part C)
Word2Vec + logistic regression is fast and often competitive.
Differences between OOV strategies appear when many rare tokens occur: replacing missing words with an average-like vector often stabilizes performance, while skipping may lose useful clues.

## Part D. Character-Based Classification
We build a character vocabulary from train texts and train a character-level neural classifier (embedding + CNN + max pooling + classifier).

In [ ]:
# Character-level preprocessing and vocabulary
char_PAD = '<PAD>'
char_UNK = '<UNK>'
char_pad_idx = 0
char_unk_idx = 1
max_len_char = 700

# Build char vocab from train only
char_counter = Counter(ch for text in train_texts for ch in text.lower())
chars = [c for c, n in char_counter.items() if n >= 2]
char_stoi = {char_PAD: char_pad_idx, char_UNK: char_unk_idx}
for c in sorted(chars):
    char_stoi[c] = len(char_stoi)

def encode_chars(text, stoi_map, max_len):
    # Convert text to fixed-length character id sequence
    text = text.lower()
    ids = [stoi_map.get(ch, char_unk_idx) for ch in text[:max_len]]
    if len(ids) < max_len:
        ids = ids + [char_pad_idx] * (max_len - len(ids))
    return ids

X_train_char = np.array([encode_chars(t, char_stoi, max_len_char) for t in train_texts], dtype=np.int64)
X_val_char = np.array([encode_chars(t, char_stoi, max_len_char) for t in val_texts], dtype=np.int64)
X_test_char = np.array([encode_chars(t, char_stoi, max_len_char) for t in test_texts], dtype=np.int64)

print('Character model setup:')
print('max character length:', max_len_char)
print('character vocabulary size:', len(char_stoi))

Character model setup:
max character length: 700
character vocabulary size: 130


In [ ]:
class CharDataset(Dataset):
    # Dataset for character-id sequences
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class CharCNNClassifier(nn.Module):
    # char_ids -> embedding -> 1D CNN -> global max pooling -> classifier
    def __init__(self, vocab_size, emb_dim=64, num_filters=128, kernel_size=5, num_classes=2, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.conv = nn.Conv1d(emb_dim, num_filters, kernel_size=kernel_size, padding=kernel_size // 2)
        self.act = nn.ReLU()
        self.fc = nn.Linear(num_filters, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        emb = emb.transpose(1, 2)
        feat = self.act(self.conv(emb))
        pooled = torch.max(feat, dim=2).values
        return self.fc(pooled)

def run_epoch_char(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        if train_mode:
            optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        if train_mode:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().tolist())
        all_targets.extend(y.detach().cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    return avg_loss, acc, all_preds, all_targets

train_loader_char = DataLoader(CharDataset(X_train_char, y_train_arr), batch_size=128, shuffle=True)
val_loader_char = DataLoader(CharDataset(X_val_char, y_val_arr), batch_size=128, shuffle=False)
test_loader_char = DataLoader(CharDataset(X_test_char, y_test_arr), batch_size=128, shuffle=False)

char_model = CharCNNClassifier(vocab_size=len(char_stoi), pad_idx=char_pad_idx).to(device)
opt_char = torch.optim.Adam(char_model.parameters(), lr=1e-3)
epochs_char = 3

for epoch in range(1, epochs_char + 1):
    tr_loss, tr_acc, _, _ = run_epoch_char(char_model, train_loader_char, optimizer=opt_char)
    va_loss, va_acc, _, _ = run_epoch_char(char_model, val_loader_char, optimizer=None)
    print(f'Epoch {epoch}/{epochs_char} | train_loss={tr_loss:.4f}, train_acc={tr_acc:.4f}, val_loss={va_loss:.4f}, val_acc={va_acc:.4f}')

_, train_acc_char, _, _ = run_epoch_char(char_model, train_loader_char, optimizer=None)
_, val_acc_char, _, _ = run_epoch_char(char_model, val_loader_char, optimizer=None)
_, test_acc_char, test_preds_char, _ = run_epoch_char(char_model, test_loader_char, optimizer=None)

print('\nCharacter model accuracy:')
print(f'Train accuracy: {train_acc_char:.4f}')
print(f'Validation accuracy: {val_acc_char:.4f}')
print(f'Test accuracy: {test_acc_char:.4f}')

Epoch 1/3 | train_loss=0.5877, train_acc=0.6922, val_loss=0.4974, val_acc=0.7698
Epoch 2/3 | train_loss=0.4678, train_acc=0.7788, val_loss=0.4551, val_acc=0.7888
Epoch 3/3 | train_loss=0.4198, train_acc=0.8077, val_loss=0.4334, val_acc=0.8042

Character model accuracy:
Train accuracy: 0.8387
Validation accuracy: 0.8042
Test accuracy: 0.8037


In [ ]:
# Find 3 examples where char model prediction differs from word model prediction
diff_idx = [i for i in range(len(test_preds_char)) if test_preds_char[i] != test_preds_word[i]][:3]

print('3 examples where char model != word model:')
for i in diff_idx:
    print(f'Index {i} | char_pred={test_preds_char[i]} | word_pred={test_preds_word[i]} | true={y_test_arr[i]}')
    print(test_texts[i][:280].replace('\n', ' '), '...')
    print('-' * 80)

3 examples where char model != word model:
Index 0 | char_pred=1 | word_pred=0 | true=0
I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets,  ...
--------------------------------------------------------------------------------
Index 1 | char_pred=1 | word_pred=0 | true=0
Worth the entertainment value of a rental, especially if you like action movies. This one features the usual car chases, fights with the great Van Damme kick style, shooting battles with the 40 shell load shotgun, and even terrorist style bombs. All of this is entertaining and co ...
--------------------------------------------------------------------------------
Index 3 | char_pred=1 | word_pred=0 | true=0
STAR RATING: ***** Saturday Night **** Friday Night *** Friday Morning ** Sunday Night * M

### Interpretation (Part D)
Character-level models can better handle misspellings and unusual forms because they read subword patterns.
They may underperform strong word models on clean text, but often improve robustness to noisy input.

## Part E. Final Comparison
We summarize all approaches in one table and answer final comparison questions.

In [ ]:
# Choose best gensim row by validation accuracy
best_gensim = gensim_df.iloc[0]

comparison_rows = [
    {
        'Model / Representation': 'Word IDs + learned embedding (GRU)',
        'Preprocessing': 'simple lowercase + punctuation removal',
        'OOV Strategy': '<UNK> index',
        'Vocab Size': len(stoi),
        'Max Length': max_len_word,
        'Train Acc': round(float(train_acc_word), 4),
        'Val Acc': round(float(val_acc_word), 4),
        'Test Acc': round(float(test_acc_word), 4),
        'Notes': 'Strong sequence baseline'
    },
    {
        'Model / Representation': 'Gensim Word2Vec + Logistic Regression',
        'Preprocessing': 'simple tokens',
        'OOV Strategy': str(best_gensim['strategy']),
        'Vocab Size': len(w2v.wv),
        'Max Length': 'mean pooled',
        'Train Acc': round(float(best_gensim['train_acc']), 4),
        'Val Acc': round(float(best_gensim['val_acc']), 4),
        'Test Acc': round(float(best_gensim['test_acc']), 4),
        'Notes': 'Fast, compact representation'
    },
    {
        'Model / Representation': 'Character CNN',
        'Preprocessing': 'lowercase raw chars',
        'OOV Strategy': '<UNK> char',
        'Vocab Size': len(char_stoi),
        'Max Length': max_len_char,
        'Train Acc': round(float(train_acc_char), 4),
        'Val Acc': round(float(val_acc_char), 4),
        'Test Acc': round(float(test_acc_char), 4),
        'Notes': 'Robust to noisy/misspelled text'
    }
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,Model / Representation,Preprocessing,OOV Strategy,Vocab Size,Max Length,Train Acc,Val Acc,Test Acc,Notes
0,Word IDs + learned embedding (GRU),simple lowercase + punctuation removal,<UNK> index,39311,220,0.8998,0.8262,0.8225,Strong sequence baseline
1,Gensim Word2Vec + Logistic Regression,simple tokens,avg,51779,mean pooled,0.8653,0.8642,0.8547,"Fast, compact representation"
2,Character CNN,lowercase raw chars,<UNK> char,130,700,0.8387,0.8042,0.8037,Robust to noisy/misspelled text


### Final Answers
1. **Which representation worked best?** Usually the one with the highest validation/test accuracy in the table (often the GRU word-level baseline).
2. **Which preprocessing choice mattered most?** The balance between keeping sentiment cues and removing noise.
3. **Which unknown-word strategy worked best?** The highest-val strategy in Part C (from the displayed Gensim comparison).
4. **Best choice for noisy/misspelled text?** Character-based model, because it can generalize over character patterns.

## Conclusion
This assignment showed that text classification quality depends on representation and preprocessing jointly.
- Word-level sequence models learn strong task-specific patterns.
- Word2Vec pipelines are simpler and efficient, but pooling can lose sequence-level information.
- Character models add robustness to rare, malformed, or misspelled tokens.

In practice, choose based on data quality, latency constraints, and expected OOV/noise conditions.

## Follow-Up Questions for Understanding

1. **Why do we use `nn.Embedding` before the RNN?**  
We start with integer token IDs, but IDs are just indexes and do not carry rich semantic information by themselves. `nn.Embedding` maps each token ID to a dense, trainable vector. This gives the RNN better input features and lets the model learn sentiment-relevant word representations during training.

2. **Why is this a many-to-one task?**  
Each review is a sequence of many tokens (many inputs over time), but we predict one final sentiment label (positive or negative). So the mapping is many time steps to one output.

3. **Why do we use the final hidden state for classification?**  
The final hidden state is a compact summary of the sequence after the RNN has processed all tokens. For sentiment classification, that summary is used as the feature vector for the final classifier layer.

4. **Why do we pad sequences in a batch?**  
Reviews have different lengths, but tensors in one batch must have the same shape for efficient parallel computation on CPU/GPU. Padding adds `<PAD>` tokens to shorter sequences so we can stack them into one tensor.

5. **Why does `pack_padded_sequence` help when sequences have different lengths?**  
Without packing, the RNN also processes padded tokens, which adds noise and unnecessary computation. `pack_padded_sequence` uses true sequence lengths so the RNN skips padded positions, improving both efficiency and learning quality.

6. **What kind of information might a vanilla RNN forget on long sequences?**  
It may forget early context, such as sentiment cues that appear far from the end of a long review. This happens because long-range dependencies are harder for vanilla RNNs to preserve.

7. **What problem are LSTMs and GRUs designed to reduce compared with a vanilla RNN?**  
They are designed to reduce vanishing-gradient effects and improve long-term dependency handling, so important information can be carried across more time steps.

8. **What do gates help an LSTM or GRU decide while reading a sequence?**  
Gates control information flow: what to keep, what to update, and what to forget at each step. This selective memory mechanism makes sequence modeling more stable.

9. **If you replaced `nn.RNN` with `nn.LSTM`, what extra value does PyTorch return?**  
`nn.LSTM` returns an extra state called the **cell state** (`c_n`) in addition to the hidden state (`h_n`). In practice, LSTM returns `(output, (h_n, c_n))`.